# Environment check — the uv project

Question 7: verify that a notebook running in this project's `uv`
environment can import what `uv add` installed.

Launch it the way the handout does, which adds Jupyter for the run
without recording it as a project dependency:

```bash
uv run --with jupyter jupyter lab
```


## Which interpreter is this?

Not the project's `.venv`, which is the surprise. `uv run --with jupyter`
builds an **ephemeral overlay environment** under `~/.cache/uv/` that layers
Jupyter on top of the project's dependencies, and runs there. The handout
describes this as launching "with the virtual environment activated"; what
actually happens is closer to a throwaway copy that can see the project.

The test is not the path, then, but whether the project's packages resolve.


In [1]:
import sys

print('executable:', sys.executable)
print('prefix    :', sys.prefix)


executable: ~/.cache/uv/builds-v0/.tmpcSm9cf/bin/python
prefix    : ~/.cache/uv/builds-v0/.tmpcSm9cf


## The declared dependency

`requests` is the one package in `pyproject.toml`.

In [2]:
import requests

print('requests', requests.__version__)
print('loaded from', requests.__file__)


requests 2.34.2
loaded from ~/.cache/uv/archive-v0/-D7cKeKbP_BuNpXr/lib/python3.12/site-packages/requests/__init__.py


## Jupyter itself is *not* a dependency

`--with` puts it in a temporary overlay, so it is importable here
while `pyproject.toml` stays clean. That is the distinction the
single flat `requirements.txt` of exercise 1 cannot express.

In [3]:
import tomllib
from pathlib import Path

declared = tomllib.loads(Path('pyproject.toml').read_text())
print('runtime :', declared['project']['dependencies'])
print('dev     :', declared['dependency-groups']['dev'])

runtime : ['requests>=2.34.2']
dev     : ['mypy>=2.3.1', 'pylint>=4.0.8', 'pytest>=9.1.1', 'types-requests>=2.33.0.20260906']


## The project's own modules import too

Not just installed packages: the notebook sits inside the project,
so the code written for questions 5 and 6 is importable as well.

In [4]:
from current_weather import query_current_weather
from locations import FRANCE

grenoble = query_current_weather(*FRANCE['Grenoble'])['current']
print('Grenoble now:', grenoble['temperature_2m'], '°C, WMO', grenoble['weather_code'])

Grenoble now: 19.3 °C, WMO 3
